In [1]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import json
import time
from pathlib import Path
import pandas as pd

# Load facts
facts = []
with open("../data/facts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        facts.append(json.loads(line))

facts_df = pd.DataFrame(facts)
print(f"Loaded {len(facts_df)} facts from KG")

# Load raw text for context retrieval
raw_texts = []
if Path("../data/raw_text.jsonl").exists():
    with open("../data/raw_text.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            raw_texts.append(json.loads(line))
    raw_text_df = pd.DataFrame(raw_texts)
    print(f"Loaded {len(raw_text_df)} text chunks")

Loaded 485 facts from KG
Loaded 134 text chunks


In [3]:
def predicate_to_verb(predicate):
    mapping = {
        "hasNutrient": "contains",
        "usesTechnique": "is prepared using",
        "hasGuideline": "follows the guideline",
        "recommendsTechnique": "recommends",
        "aimsToImprove": "aims to improve",
        "affectsRiskOf": "affects the risk of",
        "associatedWithOutcome": "is associated with",
        "hasEnvironmentalImpact": "has an environmental impact",
        "guidelineTargetsImpact": "targets environmental impact",
        "affectsImpactCategory": "affects the environmental impact category"
    }
    return mapping.get(predicate, predicate)

In [4]:
term_conversion = {
        "nutrient" : ["hasNutrient"],
        "contain" : ["hasNutrient"],
        "use" : ["usesTechnique", "hasNutrient"],
        "rich" : ["hasNutrient"],
        "find" : ["hasNutrient"],
        "ingredient" : ["hasNutrient"],
        "follow" : ["usesTechnique", "hasGuideline"],
        "affect" : ["hasEnvironmentalImpact", "affectsRiskOf", "guidelineTargetsImpact", "affectsImpactCategory"],
        "risk" : ["affectRiskOf"],
        "associate" : ["associatedWithOutcome", "hasNutrient", "usesTechnique"],
        "outcome" : ["associatedWithOutcome"],
        "recommend" : ["recommendsTechnique"],
        "environment" : ["hasEnvironmentalImpact", "guidelineTargetsImpact", "affectsImpactCategory"],
        "impact" : ["hasEnvironmentalImpact", "affectsRiskOf", "guidelineTargetsImpact", "affectsImpactCategory"],
        "prepare" : ["usesTechnique", "hasGuideline"],
        "guideline" : ["hasGuideline", "guidelineTargetsImpact"],
        "aim" : ["aimsToImprove", "hasGuideline", "guidelineTargetsImpact"],
        "improve" : ["aimsToImprove", "hasGuideline"],
        "target" : ["guidelineTargetsImpact"],
        "sustain" : ["guidelineTargetsImpact", "aimsToImprove", "hasEnvironmentalImpact", "affectsImpactCategory"],
        "benefit" : ["usesTechnique", "hasNutrient", "aimsToImprove", "hasGuideline", "hasEnvironmentalImpact", "affectsRiskOf", "guidelineTargetsImpact", "affectsImpactCategory"],
        "dietary": ["hasNutrient", "hasGuideline"],
        "advice": ["hasGuideline"],
        "preparation": ["usesTechnique", "hasNutrient"],
        "method": ["usesTechnique", "associatedWithOutcome"],
        "cook": ["usesTechnique", "associatedWithOutcome", "hasEnvironmentalImpact"],
        "process": ["usesTechnique", "hasNutrient", "associatedWithOutcome"]
    }

def predicate_to_search(query_terms):
    pred = []
    for term in query_terms:
        if term in term_conversion.keys():
            for p in term_conversion[term]:
                if p not in pred:
                    pred.append(p)
    return pred

In [5]:
from Levenshtein import distance as levenshtein
import spacy
nlp = spacy.load("en_core_web_sm")

def fuzzy_match(df, term, predicate, edit=True):
    """Return rows where subject/object matches term exactly or within edit distance ≤ 1."""
    
    def check_row(row):
        subj = str(row['subject']).lower()
        obj = str(row['object']).lower()
        t   = term.lower()

        # Exact match first (fast)
        if subj == t or obj == t:
            return True
        if edit: 
            # Fuzzy match (edit distance ≤ 1)
            if levenshtein(subj, t) <= 1 or levenshtein(obj, t) <= 1:
                return True
        
        return False

    # Filter rows by predicate + fuzzy function
    filtered = df[df['predicate'] == predicate]
    return filtered[filtered.apply(check_row, axis=1)]


def retrieve_from_kg(unigram, bigram = None):
    """
    Retrieve relevant facts from KG based on query terms.
    Uses simple keyword matching across subject, predicate, object.
    """
    results = []
    pred = predicate_to_search(unigram)
    for p in pred:
        if bigram:
            for terms in bigram:
                # Search in subject, predicate, or object
                matches = fuzzy_match(facts_df, terms, p, edit=False)
                if not matches.empty:
                    token_remove = terms.split("_")
                    unigram = [u for u in unigram if u not in token_remove]
                results.append(matches)
        if unigram:    
            for term in unigram:
                # Search in subject, predicate, or object
                matches = fuzzy_match(facts_df, term, p)
                results.append(matches)
    if results:
        combined = pd.concat(results)
        combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
        combined = combined.drop_duplicates()
        return combined

    return pd.DataFrame()


def get_text_context(fact_pages, top_k=100):
    """
    Retrieve original text chunks associated with the pages in retrieved facts.
    """
    if not raw_texts:
        return []
    
    context_chunks = []
    for page in fact_pages:
        chunks = [t for t in raw_texts if t.get('page') == page]
        context_chunks.extend(chunks)
    
    return context_chunks[:top_k]


def extract_query_terms(question):
    """
    Extraction of key terms from question.
    """

    doc = nlp(question.lower())

    unigram = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]

    bigram = ["_".join(unigram[i:i+2]) for i in range(len(unigram)-1)]
    
    return unigram, bigram 

In [6]:
import os
from groq import Groq

# Initialize client
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

In [ ]:
def rag_kg_baseline(question, use_chunks=False, pred_conversion=True):
    """
    RAG over Knowledge Graph baseline.
    Retrieves facts from KG, builds context, and generates answer with LLM.
    
    Returns:
        - answer: Generated text
        - metadata: dict with context, facts, latency, context_length
    """
    start_time = time.time()
    
    # Extract query terms and retrieve from KG
    unigram, bigram = extract_query_terms(question)
    retrieved_facts = retrieve_from_kg(unigram, bigram)
    
    # Build context from retrieved facts
    context_parts = []
    pages_cited = []
    contexts = {}

    if not retrieved_facts.empty:
        context_parts.append("Relevant Knowledge Graph Facts:")
        for idx, row in retrieved_facts.iterrows():
            key = (row['subject'], row['predicate'], row['object'])
            page = row.get('page')
            if key not in contexts:
                contexts[key] = [] 
            if pd.notna(page):
                page = int(page)
                if page not in contexts[key]:
                    contexts[key].append(page)
                    pages_cited.append(page)

        for (subj, pred, obj), pages in contexts.items():
            verb = predicate_to_verb(pred)
            if pred_conversion:
                base = f"- {subj} {verb} {obj}"
            else:
                base = f"- {subj} {pred} {obj}"
            if pages:
                page_text = ", ".join(str(p) for p in sorted(pages))
                base += f" (Page {page_text})"
            context_parts.append(base)
        
        # Optionally retrieve original text chunks
        if use_chunks:
            if raw_texts:
                text_chunks = get_text_context(pages_cited)
                if text_chunks:
                    context_parts.append("\nRelevant Text Excerpts:")
                    for chunk in text_chunks:
                        excerpt = chunk.get('text', '')[:200]  # Limit length
                        page = chunk.get('page', 'N/A')
                        context_parts.append(f"[Page {page}] {excerpt}...")
    
    context = "\n".join(context_parts)
    # Build RAG prompt
    prompt = f"""Based on the following information from a food science handbook:

{context}

Please answer this question: {question}

Provide a clear answer and cite the page numbers where the information comes from."""
    
    # Generate with LLM
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"Error generating response: {e}"
    
    latency = time.time() - start_time
    
    metadata = {
        "method": "RAG-KG",
        "question": question,
        "retrieved_facts": retrieved_facts.to_dict('records') if not retrieved_facts.empty else [],
        "context": context,
        "context_length": len(prompt),
        "latency_seconds": latency,
        "pages_cited": sorted(set(pages_cited))
    }
    
    return answer, metadata

In [ ]:
def zero_shot_baseline(question):
    """
    Zero-shot LLM baseline.
    Sends question directly to LLM without any KG retrieval or grounding.
    
    Returns:
        - answer: Generated text
        - metadata: dict with latency, context_length
    """
    start_time = time.time()
    
    # Simple prompt with no additional context
    prompt = f"Answer this question: {question}"
    
    # Generate with LLM
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"Error generating response: {e}"
    
    latency = time.time() - start_time
    
    metadata = {
        "method": "Zero-Shot",
        "question": question,
        "context": None,
        "context_length": len(prompt),
        "latency_seconds": latency,
        "pages_cited": []
    }
    
    return answer, metadata